# 🎮 Steam Review Success Prediction

In [ ]:
#Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

# Save models
import joblib

In [ ]:
df = pd.read_csv("../data/games.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df["release_date"] = pd.to_datetime(df["release_date"])

df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month

In [ ]:
review_ratio = df["positive"] / (df["positive"] + df["negative"])

df["review_success"] = (review_ratio >= 0.80).astype(int)

In [ ]:
df["review_success"].value_counts()

In [ ]:
df["review_success"].value_counts(normalize=True)

In [ ]:
columns_to_drop = [
    "appid",
    "name",

    "detailed_description",
    "about_the_game",
    "short_description",
    "reviews",

    "header_image",
    "website",
    "support_url",
    "support_email",
    "metacritic_url",

    "screenshots",
    "movies",

    "notes",
    "packages",

    "positive",
    "negative",
    "pct_pos_total",
    "pct_pos_recent",
    "num_reviews_total",
    "num_reviews_recent",

    "score_rank",
    "user_score",
    "metacritic_score",

    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",

    "peak_ccu",
    "estimated_owners"
]

In [ ]:
target_counts = df["review_success"].value_counts()

plt.figure(figsize=(6, 5))

plt.bar(
    target_counts.index.astype(str),
    target_counts.values
)

plt.title("Review Success Distribution")
plt.xlabel("Review Success")
plt.ylabel("Number of Games")

plt.tight_layout()

plt.savefig("../screenshots/class_distribution.png", dpi=300)

plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns

numeric_columns

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "bool"]).columns

categorical_columns

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
correlation = df.select_dtypes(include=["number"]).corr()

plt.figure(figsize=(12, 10))

plt.imshow(correlation)

plt.colorbar()

plt.xticks(
    range(len(correlation.columns)),
    correlation.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation.columns)),
    correlation.columns
)

plt.tight_layout()

plt.savefig("../screenshots/correlation_matrix.png", dpi=300)

plt.show()

In [ ]:
X = df.drop(columns=["review_success"])
y = df["review_success"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
columns_to_drop = [
    "appid",
    "name",
    "detailed_description",
    "about_the_game",
    "short_description",
    "reviews",
    "header_image",
    "website",
    "support_url",
    "support_email",
    "metacritic_url",
    "notes",
    "screenshots",
    "movies",
    "packages",

    # Leakage
    "positive",
    "negative",
    "pct_pos_total",
    "pct_pos_recent",
    "num_reviews_total",
    "num_reviews_recent",

    # Future information
    "score_rank",
    "user_score",
    "metacritic_score",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "peak_ccu",
    "estimated_owners",

    # We already extracted year & month
    "release_date"
]

df = df.drop(columns=columns_to_drop)

In [ ]:
df.info()

In [ ]:
X = df.drop(columns=["review_success"])
y = df["review_success"]

In [ ]:
df["review_success"].value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

## Preprocessing Pipeline

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object", "bool"]
).columns

print(numeric_features)
print(categorical_features)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

## Logistic Regression

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            random_state=42,
            max_iter=1000
        ))
    ]
)

In [ ]:
logistic_model.fit(X_train, y_train)

In [ ]:
y_pred = logistic_model.predict(X_test)

In [ ]:
y_prob = logistic_model.predict_proba(X_test)

In [ ]:
y_prob[:5]

In [ ]:
success_probability = y_prob[:, 1]

In [ ]:
#compare
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability": success_probability
})

comparison.head(10)

## Classification Metrics

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot()

plt.title("Confusion Matrix")

plt.tight_layout()

plt.savefig("../screenshots/confusion_matrix.png", dpi=300)

plt.show()

## Decision Tree model

In [ ]:
decision_tree = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            random_state=42,
            max_depth=10
        ))
    ]
)

In [ ]:
decision_tree.fit(X_train, y_train)

In [ ]:
dt_pred = decision_tree.predict(X_test)

In [ ]:
print("Accuracy :", accuracy_score(y_test, dt_pred))
print("Precision:", precision_score(y_test, dt_pred))
print("Recall   :", recall_score(y_test, dt_pred))
print("F1 Score :", f1_score(y_test, dt_pred))

## Random Forest Model

In [ ]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

In [ ]:
random_forest.fit(X_train, y_train)

In [ ]:
rf_pred = random_forest.predict(X_test)

In [ ]:
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))

## ROC Curve & ROC-AUC

In [ ]:
lr_prob = logistic_model.predict_proba(X_test)[:, 1]

dt_prob = decision_tree.predict_proba(X_test)[:, 1]

rf_prob = random_forest.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import roc_curve

In [ ]:
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)

dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_prob)

rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)

In [ ]:
from sklearn.metrics import roc_auc_score

lr_auc = roc_auc_score(y_test, lr_prob)

dt_auc = roc_auc_score(y_test, dt_prob)

rf_auc = roc_auc_score(y_test, rf_prob)

print("Logistic :", lr_auc)
print("Tree     :", dt_auc)
print("Forest   :", rf_auc)

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(
    lr_fpr,
    lr_tpr,
    label=f"Logistic Regression (AUC = {lr_auc:.3f})"
)

plt.plot(
    dt_fpr,
    dt_tpr,
    label=f"Decision Tree (AUC = {dt_auc:.3f})"
)

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_auc:.3f})"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    label="Random Guess"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig("../screenshots/roc_curve_models.png", dpi=300)

plt.show()

## Threshold Tuning

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.30, 0.50, 0.70]

results = []

for threshold in thresholds:

    predictions = (lr_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions)
    })

threshold_results = pd.DataFrame(results)

threshold_results

In [ ]:
threshold_results.set_index("Threshold").plot(
    figsize=(8,5),
    marker="o"
)

plt.title("Threshold Tuning")

plt.ylabel("Score")

plt.grid(True)

plt.tight_layout()

plt.savefig("../screenshots/threshold_tuning.png", dpi=300)

plt.show()

## Feature Importance

In [ ]:
rf_model = random_forest.named_steps["classifier"]

In [ ]:
feature_names = random_forest.named_steps[
    "preprocessor"
].get_feature_names_out()

In [ ]:
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(15)

In [ ]:
top_features = importance_df.head(15)

plt.figure(figsize=(10,6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.gca().invert_yaxis()

plt.title("Top 15 Most Important Features")

plt.xlabel("Importance")

plt.tight_layout()

plt.savefig(
    "../screenshots/feature_importance.png",
    dpi=300
)

plt.show()

## Model Comparison Dashboard

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        0.7214,
        0.6644,
        0.6993
    ],
    "Precision": [
        0.6612,
        0.6090,
        0.6628
    ],
    "Recall": [
        0.7287,
        0.6232,
        0.6192
    ],
    "F1 Score": [
        0.6933,
        0.6160,
        0.6402
    ],
    "ROC-AUC": [
        0.8079,
        0.7440,
        0.7827
    ]
})

comparison_df

In [ ]:
metrics = comparison_df.columns[1:]

x = np.arange(len(metrics))

width = 0.25

plt.figure(figsize=(11,6))

plt.bar(
    x - width,
    comparison_df.iloc[0,1:],
    width,
    label="Logistic Regression"
)

plt.bar(
    x,
    comparison_df.iloc[1,1:],
    width,
    label="Decision Tree"
)

plt.bar(
    x + width,
    comparison_df.iloc[2,1:],
    width,
    label="Random Forest"
)

plt.xticks(x, metrics)

plt.ylim(0,1)

plt.ylabel("Score")

plt.title("Model Performance Comparison")

plt.legend()

plt.grid(axis="y")

plt.tight_layout()

plt.savefig(
    "../screenshots/model_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
comparison_df.to_csv(
    "../results/model_comparison.csv",
    index=False
)

In [ ]:
joblib.dump(
    logistic_model,
    "../models/logistic_regression.pkl"
)

In [ ]:
joblib.dump(
    decision_tree,
    "../models/decision_tree.pkl"
)

In [ ]:
joblib.dump(
    random_forest,
    "../models/random_forest.pkl"
)